### **Подключение модулей**

In [ ]:
from system_check import *
CUDA_state_print()

In [ ]:
import sys
# sys.path.append("/home/pmartynyuk/UnTIE project/scripts/models_processing")
# sys.path.append("/home/pmartynyuk/UnTIE project/scripts/text_processing")
# sys.path.append("/home/pmartynyuk/UnTIE project/scripts/classes")
# sys.path.append("/home/pmartynyuk/.local/bin")


from models_processing.models_setting import *
from text_processing.text_extraction import *
from classes.model_classes import *

# from langchain.vectorstores import FAISS

### **Подключение датасета**

In [ ]:
import pandas as pd

df_loaded = pd.read_json("../datasets/scirex_structured.json", orient="records", lines=True)
df_loaded

### Номер примера

In [ ]:
num_of_sample = 0

In [ ]:
df_loaded["tasks"][num_of_sample]

In [ ]:
text = df_loaded["original_text"][num_of_sample]

In [ ]:
print(text)

### Разбивка на предложения и чанки

In [ ]:
text_proc = TextProcesser()
sents = text_proc.split_into_sentences(text)
my_sentences = [Sentence(sents[i], i) for i in range(len(sents))]
len(my_sentences)

In [ ]:
chunk_builder = ChunkBuilder()
text_chunks = chunk_builder.build_chunks(my_sentences)
len(text_chunks)

In [ ]:
for t_ch in text_chunks:
    print(t_ch)

### Вопросы

In [ ]:
question1 = Question("Which task was solved?")
question2 = Question("What problem was being solved?")
question3 = Question("What is the main task?")

questions = [question1]

### Фильтрация чанков

In [ ]:
ch_filter = ChunkFilter("simple cosine")

relevant_chunks = ch_filter.filter_chunks(question1, text_chunks)


In [ ]:
len(text_chunks)

In [ ]:
len(relevant_chunks)

In [ ]:
text_chunks

In [ ]:
relevant_chunks

In [ ]:
simp_ans_finder = SimpleAnswerFinder()

In [ ]:
questions_with_answers = []

for question in questions:
    question_with_answers = simp_ans_finder.find_answers(
    question = question,
    chunks = text_chunks
    )
    questions_with_answers.append(question_with_answers)

In [ ]:
print("Founded answers:")
for question_with_answers in questions_with_answers:
    print(f"--->{questions_with_answers.index(question_with_answers) + 1}/{len(questions_with_answers)}")
    print(f"--->{question_with_answers.text}")
    for ans in question_with_answers.answers:
        print(ans.text)
        print(ans.confidence)
    print(f"------------------------")

In [ ]:
them_asp = ThematicAspect("Task", questions_with_answers)

In [ ]:
ans_agg = AnswerAggregator()

In [ ]:
fin_ans = ans_agg.aggregate_with_clustering(them_asp)

In [ ]:
fin_ans

In [ ]:
fin_ans = ans_agg.aggregate_without_clustering(them_asp)

In [ ]:
fin_ans

In [ ]:
validator = AnswerValidator()

answers = them_asp.get_all_answers()

reference = df_loaded["tasks"][num_of_sample][0]
reference = reference.replace("_", " ")
reference

In [ ]:
# Обработка
result = validator.process(answers, reference)

In [ ]:
result

In [ ]:
print(result.keys())

In [ ]:
ch_filter = ChunkFilter()

rel_chunks = ch_filter.filter_chunks_by_keywords(text_chunks, result["analysis"]["keywords"])

rel_chunks

In [ ]:
for chunk in rel_chunks:
    print(chunk.text)
    print("***")

In [ ]:
len(rel_chunks)

In [ ]:
simp_ans_finder = SimpleAnswerFinder()

questions_with_answers = []

for question in questions:
    question.clear_answers()
    question_with_answers = simp_ans_finder.find_answers(
    question = question,
    chunks = rel_chunks
    )
    questions_with_answers.append(question_with_answers)

print("Number of questions:")
print(len(questions_with_answers))
for question in questions_with_answers:
    print("Number of answers:")
    print(len(question.answers))

print("Founded answers:")
for question_with_answers in questions_with_answers:
    print(f"--->{questions_with_answers.index(question_with_answers) + 1}/{len(questions_with_answers)}")
    print(f"--->{question_with_answers.text}")
    for ans in question_with_answers.answers:
        print(ans.text)
        print(ans.confidence)
    print(f"------------------------")

them_asp = ThematicAspect("Task", questions_with_answers)
ans_agg = AnswerAggregator()
fin_ans = ans_agg.aggregate_without_clustering(them_asp)
fin_ans

In [ ]:
fin_ans = ans_agg.aggregate_with_clustering(them_asp)
fin_ans

In [ ]:
fin_ans.supporting_answers

### Для релевантных чанков

In [ ]:
relevant_chunks = [r_ch[0] for r_ch in relevant_chunks]

In [ ]:
simp_ans_finder = SimpleAnswerFinder()

questions_with_answers = []

for question in questions:
    question_with_answers = simp_ans_finder.find_answers(
    question = question,
    chunks = relevant_chunks
    )
    questions_with_answers.append(question_with_answers)

print("Founded answers:")
for question_with_answers in questions_with_answers:
    print(f"--->{questions_with_answers.index(question_with_answers) + 1}/{len(questions_with_answers)}")
    print(f"--->{question_with_answers.text}")
    for ans in question_with_answers.answers:
        print(ans.text)
        print(ans.confidence)
    print(f"------------------------")

them_asp = ThematicAspect("Task", questions_with_answers)
ans_agg = AnswerAggregator()
fin_ans = ans_agg.aggregate_without_clustering(them_asp)
fin_ans